In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import matplotlib.pyplot as plt
from fcmeans import FCM

parent_dir = str(Path.cwd().resolve().parent)

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import experiment_common as ec

ec = importlib.reload(ec)

FREQUENCY = 150
SNR_DB = -7
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

noisy, clean, metadata = ec.simulate(waveform="ricker",frequency=FREQUENCY,snr_db=SNR_DB,noise_type="WGN",)
true_arrival = int(metadata["true_arrival"])
true_peak = int(metadata["true_peak_sample"])

print("========================================")
print("Synthetic signal")
print("========================================")
print(f"Frequency       : {FREQUENCY} Hz")
print(f"Target SNR      : {SNR_DB} dB")
print(f"Samples         : {len(noisy)}")
print(f"True arrival    : {true_arrival} sample")
print(f"True peak       : {true_peak} sample")
print()

enhanced, kept_idx, freqs = ec.cwt_hos_icwt(noisy)
if kept_idx.size == 0:
    raise RuntimeError("HOS preprocessing did not retain any CWT scale.")
print("========================================")
print("CWT-HOS-iCWT")
print("========================================")
print(f"Enhanced samples : {len(enhanced)}")
print(f"Retained scales  : {len(kept_idx)}")
print()

features_std = ec.build_feature_matrix(enhanced,feature_names=("M", "STD"),window_size=ec.WINDOW_SIZE,)
if features_std is None:
    raise RuntimeError("M + P + std feature matrix is invalid.")
print("========================================")
print("Feature settings")
print("========================================")
print(f"WINDOW_SIZE : {ec.WINDOW_SIZE}")
print()
print(f"M + P + std : {features_std.shape}")
print()

def run_basic_fcm(features, n_clusters):
    model = FCM(n_clusters=n_clusters,max_iter=ec.FCM_MAX_ITER,m=ec.FCM_M,error=ec.FCM_ERROR,random_state=ec.FCM_RANDOM_STATE,)
    model.fit(features)
    centers = np.asarray(model.centers)
    membership = np.asarray(model.u)
    signal_cluster = int(
        np.argmax(centers.mean(axis=1))
    )
    signal_membership = membership[:, signal_cluster]
    return {
        "model": model,
        "centers": centers,
        "membership": membership,
        "signal_cluster": signal_cluster,
        "signal_membership": signal_membership,
    }

result_std_c2 = run_basic_fcm(
    features_std,
    n_clusters=2,
)

result_std_c3 = run_basic_fcm(
    features_std,
    n_clusters=3,
)
membership_std_c2 = result_std_c2["signal_membership"]
membership_std_c3 = result_std_c3["signal_membership"]
print("========================================")
print("Basic FCM results")
print("========================================")
print(
    "M + Std,  C=2 -> signal cluster =",
    result_std_c2["signal_cluster"]
)
print(
    "M + Std,  C=3 -> signal cluster =",
    result_std_c3["signal_cluster"]
)
print()

M_std = features_std[:, 0]
STD = features_std[:, 1]

time_ms = np.arange(len(noisy)) / ec.FS * 1000.0
true_arrival_ms = true_arrival / ec.FS * 1000.0

def plot_feature_membership(ax,time_ms,feature1,feature2,membership,panel_label,cluster_num,):

    ax.plot(time_ms,feature1,lw=1.0,label="M",)
    ax.plot(time_ms,feature2,lw=1.0,label="Std",)

    ax.plot(time_ms,membership,color="black",lw=1.1,ls="--",label="Membership",)

    ax.axvline(
        true_arrival_ms,color="black",lw=0.8,ls=":",alpha=0.6,)

    ax.set_xlim(0, 450)
    ax.set_ylim(-0.03, 1.05)

    ax.set_ylabel("Normalized value")

    ax.text(0.015,0.88,panel_label,transform=ax.transAxes,fontsize=12,)

    ax.text(0.985,0.88,f"C = {cluster_num}",transform=ax.transAxes,ha="right",fontsize=10,)

    ax.legend(loc="upper right",frameon=False,ncol=4,fontsize=8.5,bbox_to_anchor=(0.99, 0.76),
    )

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 11,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
})

fig, axes = plt.subplots(2,1,figsize=(10, 4.0),sharex=True,)

ax = axes[0]

ax.plot(time_ms,noisy,color="black",lw=0.8,label="Noisy waveform",)

ax.axvline(true_arrival_ms,color="red",ls="--",lw=0.9,label="True arrival",)

ax.set_xlim(0, 450)

ax.set_ylabel("Amplitude")

ax.text(0.015,0.88,"(a)",transform=ax.transAxes,fontsize=12,)

ax.text(0.985,0.88,"100 Hz, = -9 dB",transform=ax.transAxes,ha="right",fontsize=10,)

ax.legend(loc="upper right",frameon=False,fontsize=9,bbox_to_anchor=(0.99, 0.76),)

plot_feature_membership(
    ax=axes[1],
    time_ms=time_ms,
    feature1=M_std,
    feature2=STD,
    membership=membership_std_c2,
    panel_label="(b)",
    cluster_num=2,
)

for ax in axes:
    ax.set_xlim(0, 450)

plt.subplots_adjust(
    left=0.11,
    right=0.98,
    bottom=0.06,
    top=0.99,
    hspace=0.22,
)

plt.show()
